# Install XGBoost

In [ ]:
import sys
!"{sys.executable}" -m pip install xgboost

In [ ]:
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import os

os.makedirs('data/models', exist_ok=True)

# ── Load datasets ──────────────────────────────────────────────
datasets = {
    'smote': {
        'X_train': joblib.load('data/features/X_train_resampled.pkl'),
        'y_train': joblib.load('data/smote/y_train_resampled.pkl'),
    },
    'under': {
        'X_train': joblib.load('data/features/X_train_under_eng.pkl'),
        'y_train': joblib.load('data/smote/y_train_under.pkl'),
    },
}

# Shared test set (same for both)
X_test  = joblib.load('data/features/X_test_eng.pkl')
y_test  = joblib.load('data/preprocessed/y_test.pkl')

# ── Model definitions ──────────────────────────────────────────
def get_models():
    return {
        'logreg': LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ),
        'xgboost': XGBClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            min_child_weight=5,   # regularization — important for small datasets
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=1,   # already balanced so keep at 1
            random_state=42,
            eval_metric='logloss',
            verbosity=0
        ),
    }

# ── Train loop ─────────────────────────────────────────────────
results = {}

for sampling_name, data in datasets.items():
    X_train = data['X_train']
    y_train = data['y_train']

    print(f"\n{'='*50}")
    print(f"Training on: {sampling_name.upper()} | Train size: {X_train.shape}")
    print(f"Class balance: {dict(y_train.value_counts())}")

    for model_name, model in get_models().items():
        print(f"  Fitting {model_name}...", end=" ")
        model.fit(X_train, y_train)

        # Save model
        path = f'data/models/{model_name}_{sampling_name}.pkl'
        joblib.dump(model, path)
        print(f"saved → {path}")

        results[f"{model_name}_{sampling_name}"] = model

print(f"\n✓ All models trained. Files in data/models/:")
for f in os.listdir('data/models'):
    print(f"  {f}")